# Diffusers 원본 Anima · ComfyUI 비교용 기록

Colab **T4 GPU**에서 순서대로 실행합니다. 설정은 저장소의 JSON, 설치·생성·기록은 Python 파일을 사용합니다.

- 생성 HP: `config/generation.json`
- Python·torch·CUDA 빌드 기준: `config/environment.json`
- 모델 파일·revision: `config/model_source.json`
- 원본 실행 옵션·토크나이저: `diffusers/original_runtime.json`
- 원본 소스: `diffusers/upstream/` (`source_manifest.json`의 고정 커밋)
- 실행: `diffusers/run_original.py`, 관찰: `diffusers/trace_original.py`

원본 Diffusers의 토큰 처리·Qwen·DiT·VAE·CFG·Euler 연산을 사용합니다. 가중치 형식 변환에는 기존 `diffusers/convert_weights.py`와 원본 변환 함수를 사용합니다. 기록 함수는 원본 입력과 반환값을 그대로 전달합니다.
원본 시간표·난수 생성을 사용하며, FP16 계산에는 아래 잔차 패치를 적용합니다. 원본 Diffusers는 Qwen에 longest padding을 사용하고 어댑터에 마스크를 전달합니다. `align_comfy.py`는 호출하지 않습니다.

### FP16 잔차 패치와 NaN 추적
기존 실행에서 첫 DiT 출력 전체가 NaN이었습니다. FP16 범위 초과를 피하도록 잔차와 gate 곱셈·합산은 FP32, Attention·MLP·출력 투영은 FP16으로 실행합니다. 원본 Diffusers 블록에 dtype hook만 적용하며 ComfyUI 계산 코드는 호출하지 않습니다. 패치와 검사 코드는 `diffusers/fp16_debug/`에 분리했습니다. 전체 DiT FP32보다 메모리 부담이 작지만 실제 T4 모델 재실행은 확인이 필요합니다.

기본값은 `residual_fp32=True`, `nan_debug=False`입니다. `nan_debug=True`로 바꾸면 모듈별 NaN/Inf 검사를 실행합니다. 최초 이상이 발견되면 중단하고 결과 폴더의 `nan_debug/`에 위치·직전 정상 텐서 이름·이상 텐서를 저장합니다. 검사는 느리므로 정상 확인 후 `nan_debug=False`로 끌 수 있습니다. 패치를 끄고 검사만 켜면 원본의 발생 위치를 추적할 수 있습니다. 모듈 경계 검사이므로 모듈 내부의 정확한 연산까지 확정하지는 않습니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, sys
from IPython.display import Image, display

REPOSITORY = 'https://github.com/HisameOgasahara/DiffFlowDiT_test.git'
REVISION = 'main'
PROJECT = Path('/content/DiffFlowDiT_diffusers_original')
DATA = Path('/content/diffusers_original_data')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REVISION, REPOSITORY, str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], check=True)
DATA.mkdir(parents=True, exist_ok=True)

# 설치·추론의 화면 출력과 로그 파일을 함께 기록합니다.
def launch(command, log_path):
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen([str(x) for x in command], cwd=PROJECT,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            if process.wait():
                raise RuntimeError(f'실행 실패: {log_path}')

## 환경 설치

기존 환경 JSON의 Python 3.13.15·torch 2.14.0+cu130·torchvision 0.29.0+cu130을 사용합니다.
전용 가상환경은 Colab 패키지를 상속하지 않습니다. 기존 ComfyUI용 설치 패키지는 추가하지 않습니다.

In [ ]:
INSTALL_LOG = DATA / 'install.log'
launch([sys.executable, '-u', PROJECT / 'tools/setup_diffusers_original.py'], INSTALL_LOG)
PYTHON = PROJECT / '.venv-diffusers-original/bin/python'

## 설정과 모델 준비

저장소 JSON을 데이터 폴더로 복사하여 사용합니다. 아래 `hp`와 `runtime`을 수정할 수 있습니다.
Qwen·T5는 저장소에 포함된 tokenizer 파일을 읽습니다. 모델 가중치는 인증 없이 다운로드합니다. 파일 준비 스크립트는 4번과 공유하며 DiffSynth 모델 코드는 불러오지 않습니다.

In [ ]:
CONFIG = DATA / 'generation.json'
RUNTIME = DATA / 'runtime.json'
for source, target in [(PROJECT / 'config/generation.json', CONFIG),
                       (PROJECT / 'diffusers/original_runtime.json', RUNTIME)]:
    if not target.exists():
        shutil.copy2(source, target)
hp = json.loads(CONFIG.read_text(encoding='utf-8'))
runtime = json.loads(RUNTIME.read_text(encoding='utf-8'))
runtime.setdefault('residual_fp32', True)
runtime['nan_debug'] = False  # NaN 추적이 필요할 때만 True
# 수정 예: hp['steps'] = 30; runtime['nan_debug'] = False
CONFIG.write_text(json.dumps(hp, ensure_ascii=False, indent=2), encoding='utf-8')
RUNTIME.write_text(json.dumps(runtime, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'generation': hp, 'runtime': runtime}, ensure_ascii=False, indent=2))
TRACE = 'selected'  # 첫·둘째·마지막 스텝; 시간 측정만 할 때 'none'

In [ ]:
MODELS = DATA / 'models'
launch([PYTHON, '-u', PROJECT / 'diffsynth/prepare_original.py',
        '--config', CONFIG, '--runtime', RUNTIME, '--models', MODELS], DATA / 'prepare.log')

## 생성과 기록

`image.png`, `run.log`, 설정·모델 해시·시간표·환경·시간·메모리·호출 횟수를 저장합니다.
`TRACE='selected'`이면 초기 노이즈, 실제 토큰·마스크·조건, 텍스트 어댑터 출력,
positive/negative DiT 입력·예측, CFG 후 예측, 갱신 전후 latent, VAE 입력·출력을 저장합니다. 어댑터 출력에는 원본 conditioner의 padding이 포함됩니다.

비교용 latent 파일은 값 변경 없이 `B,C,T,H,W` 축으로 저장합니다.
`denoised`는 관찰값에서 계산한 FP32 진단값이며 추론에 사용하지 않습니다.
`pixels`는 실제 저장 이미지의 0~1 값이고 양자화 전 VAE 출력은 `decoded_pixels`입니다.
기록에는 CPU 복사·저장 비용이 포함됩니다. `scheduler_step_*` 시간은 스케줄러 갱신만 측정하며 DiT 계산 시간은 포함하지 않습니다. 전체 생성 시간은 `pipeline_total`, 가중치 변환 시간은 `convert_models`에 따로 기록됩니다.

In [ ]:
RUN_NAME = 'diffusers_original_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT = DATA / 'runs' / RUN_NAME
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
RUN_LOG = DATA / (RUN_NAME + '.log')
try:
    launch([PYTHON, '-u', PROJECT / 'diffusers/run_original.py', '--config', CONFIG,
            '--runtime', RUNTIME, '--models', MODELS, '--output', OUTPUT,
            '--mode', 'native', '--trace', TRACE], RUN_LOG)
finally:
    if OUTPUT.exists():
        shutil.copy2(RUN_LOG, OUTPUT / 'run.log')
if (OUTPUT / 'image.png').exists():
    display(Image(filename=str(OUTPUT / 'image.png')))
print('실행 폴더:', OUTPUT)

## 결과 이동과 ComfyUI 비교

기존 노트북과 같은 ZIP·비교기를 사용합니다. ComfyUI 결과 ZIP을 업로드하고 아래 `SELECTED`에 두 실행 폴더를 지정합니다.
원본 Diffusers와 기존 ComfyUI의 실제 sampler·시간표·토크나이저·정밀도 차이를 함께 확인합니다.
조건 길이나 배치 크기가 다르면 비교기는 `shape_mismatch`로 보고하며 값을 임의로 자르지 않습니다.

In [ ]:
# 결과 다운로드 — 기본 실행, 건너뛰려면 False로 변경
DOWNLOAD = True
if DOWNLOAD:
    from google.colab import files
    archive = shutil.make_archive(str(DATA / RUN_NAME), 'zip', OUTPUT.parent, OUTPUT.name)
    files.download(archive)

In [ ]:
# 이전 실행 결과 업로드 — 필요할 때 True로 변경
UPLOAD = False
if UPLOAD:
    from google.colab import files
    import io, zipfile
    for name, payload in files.upload().items():
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            destination = (DATA / 'runs').resolve()
            for item in archive.infolist():
                target = (destination / item.filename).resolve()
                if not target.is_relative_to(destination) or target.exists():
                    raise ValueError(f'허용되지 않거나 이미 존재하는 결과 경로: {item.filename}')
            archive.extractall(destination)

In [ ]:
# 7. 비교할 실행 폴더 선택
RUNS = DATA / 'runs'
available = sorted(p for p in RUNS.iterdir() if (p / 'metrics.json').exists())
for path in available:
    print(path.name)
SELECTED = []  # 예: ['comfyui_matched_euler_...', 'diffusers_matched_euler_...']
if len(SELECTED) >= 2:
    COMPARE = DATA / 'comparisons' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    subprocess.run([str(PYTHON), '-m', 'common.compare', *[str(RUNS / name) for name in SELECTED], '--output', str(COMPARE)], cwd=PROJECT, check=True)
    if (COMPARE / 'comparison.png').exists():
        display(Image(filename=str(COMPARE / 'comparison.png')))
    print((COMPARE / 'comparison.md').read_text(encoding='utf-8'))
else:
    print('다른 노트북도 실행한 뒤 SELECTED에 비교할 폴더 이름을 넣으세요.')